# Baseline en evaluatie

Een simpele regel als referentiepunt: de trainbare modellen moeten dit verslaan.

**LET OP: dit is DUMMY data.** De scores zeggen niets over echte modellen.

Draai eerst de pipeline vanuit de hoofdmap, zodat `outputs/candidate/` klaarstaat.

## 1. Data inladen

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

# De hoofdmap van de repo zoeken (daar staat pyproject.toml) en in sys.path zetten.
HOOFDMAP = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(HOOFDMAP))
DATA_MAP = HOOFDMAP / "outputs" / "candidate"

from src.candidate.data.labelling_rule import laad_benchmark, maak_configuratietabel, voldoende_categorieen

scenarios = pd.read_csv(DATA_MAP / "scenarios.csv")
configuraties = maak_configuratietabel(laad_benchmark(DATA_MAP / "dummy_benchmark.csv"))

with open(DATA_MAP / "split_manifest.json", encoding="utf-8") as bestand:
    manifest = json.load(bestand)

# Zelfde family_id als bij de scenario's.
configuraties["family_id"] = (
    configuraties["hardware"] + "__" + configuraties["input_length_tokens"].astype(str)
)

scenarios.head()

,scenario_id,family_id,hardware,input_length_tokens,min_macro_f1,max_latency_ms_per_doc,monthly_volume,has_energy_budget,max_energy_kwh_per_month,label,is_dummy
0,0,rtx3080__384,rtx3080,384,0.842,415.30,123624,True,0.911700,medium,True
1,1,ryzen5_3600x__384,ryzen5_3600x,384,0.875,4.43,22447,True,0.129600,geen,True
2,2,ryzen5_3600x__256,ryzen5_3600x,256,0.878,31.46,4805,False,NaN,geen,True
3,3,ryzen5_3600x__384,ryzen5_3600x,384,0.860,222.38,11577,False,NaN,medium,True
4,4,ryzen5_3600x__384,ryzen5_3600x,384,0.818,36.37,1353,True,0.002304,compact,True


## 2. Testfamilies apart zetten

De testfamilies blijven dicht tot de eindvergelijking met de modellen.

In [3]:
test_families = manifest["test_families"]

train = scenarios[~scenarios["family_id"].isin(test_families)]
train_configuraties = configuraties[~configuraties["family_id"].isin(test_families)]

print(len(train), "scenario's,", len(train_configuraties), "configuraties")

## 3. Drempels

Per hardware en categorie: de mediaan van de F1 en de latency in de trainfamilies.

In [4]:
def bepaal_drempels(configs):
    return configs.groupby(["hardware", "category"])[["macro_f1", "latency_ms_per_doc"]].median()


bepaal_drempels(train_configuraties)

macro_f1  latency_ms_per_doc
hardware     category                              
rtx3080      compact   0.838367            1.920167
             medium    0.878817            5.388333
ryzen5_3600x compact   0.840600           17.459667
             medium    0.879433           70.532333

## 4. De regel

Kies de kleinste categorie waarvan de drempels de eisen halen. Past er geen, dan `geen`.
De regel kijkt alleen naar minimale kwaliteit en maximale latency (per hardware), dus niet naar energie.

In [5]:
def voorspel(scenario, drempels):
    for categorie in ["compact", "medium"]:
        f1, latency = drempels.loc[(scenario["hardware"], categorie)]
        if scenario["min_macro_f1"] <= f1 and scenario["max_latency_ms_per_doc"] >= latency:
            return categorie
    return "geen"


# Eén voorbeeld proberen.
voorbeeld = train.iloc[0]
print(voorbeeld[["hardware", "min_macro_f1", "max_latency_ms_per_doc", "label"]])
print("voorspeld:", voorspel(voorbeeld, bepaal_drempels(train_configuraties)))

hardware                  rtx3080
min_macro_f1                0.842
max_latency_ms_per_doc      415.3
label                      medium
Name: 0, dtype: object
voorspeld: medium


## 5. Voorspellen per fold

Per fold komen de drempels uit de andere trainfamilies, dus nooit uit de fold zelf.

In [6]:
delen = []
for fold in manifest["cv_folds"]:
    in_fold = train_configuraties["family_id"].isin(fold["families"])
    drempels = bepaal_drempels(train_configuraties[~in_fold])

    validatie = train[train["family_id"].isin(fold["families"])].copy()
    validatie["voorspeld"] = [voorspel(rij, drempels) for _, rij in validatie.iterrows()]
    delen.append(validatie)

resultaat = pd.concat(delen)
resultaat[["family_id", "label", "voorspeld"]].head()

,family_id,label,voorspeld
6,rtx3080__64,medium,medium
14,rtx3080__64,compact,compact
15,ryzen5_3600x__64,medium,geen
18,rtx3080__64,medium,geen
29,rtx3080__64,compact,compact


## 6. Confusion matrix

Rijen = echt label, kolommen = voorspelling.

In [7]:
labels = ["compact", "medium", "geen"]

matrix = pd.crosstab(resultaat["label"], resultaat["voorspeld"]).reindex(
    index=labels, columns=labels, fill_value=0
)
matrix

voorspeld,compact,medium,geen
label,,,
compact,472,19,30
medium,1,352,161
geen,89,81,467


## 7. Scores

- **Macro-F1:** de F1 per klasse, gemiddeld.
- **Constraint violation:** de gekozen categorie haalt de eisen niet volgens de benchmark. Percentage van de keren dat er een categorie gekozen is.
- **Gemiste kans:** de regel zegt `geen`, terwijl er wel een categorie voldoet. Percentage van de scenario's waarin een categorie voldoet.

Een violation is niet hetzelfde als voorspelling != label: kiest de regel `medium` bij label `compact`, dan is dat alleen een violation als medium de eisen niet haalt.

In [8]:
def scores(df):
    # Macro-F1 uit de confusion matrix van deze rijen.
    matrix = pd.crosstab(df["label"], df["voorspeld"]).reindex(
        index=labels, columns=labels, fill_value=0
    )
    f1_per_klasse = []
    for klasse in labels:
        goed = matrix.loc[klasse, klasse]
        te_veel = matrix[klasse].sum() - goed
        gemist = matrix.loc[klasse].sum() - goed
        f1_per_klasse.append(2 * goed / (2 * goed + te_veel + gemist))

    # Violations: haalt de gekozen categorie de eisen niet?
    gekozen = df[df["voorspeld"] != "geen"]
    violations = sum(
        rij["voorspeld"] not in voldoende_categorieen(configuraties, rij.to_dict())
        for _, rij in gekozen.iterrows()
    )

    # Gemiste kansen: "geen" voorspeld terwijl er wel een categorie voldoet.
    kansen = df[df["label"] != "geen"]
    gemiste_kansen = (kansen["voorspeld"] == "geen").sum()

    return {
        "macro_f1": sum(f1_per_klasse) / 3,
        "violations": violations,
        "keuzes": len(gekozen),
        "violation_rate": violations / len(gekozen),
        "gemiste_kansen": gemiste_kansen,
        "kansen": len(kansen),
        "gemiste_kans_rate": gemiste_kansen / len(kansen),
    }


scores(resultaat)

{'macro_f1': np.float64(0.7738889351318594),
 'violations': 172,
 'keuzes': 1014,
 'violation_rate': 0.16962524654832348,
 'gemiste_kansen': np.int64(191),
 'kansen': 1035,
 'gemiste_kans_rate': np.float64(0.18454106280193236)}

## 8. Scores per inputlengte

Elke fold is één inputlengte.

In [9]:
pd.DataFrame(
    {lengte: scores(deel) for lengte, deel in resultaat.groupby("input_length_tokens")}
).T.round(3)

,macro_f1,violations,keuzes,violation_rate,gemiste_kansen,kansen,gemiste_kans_rate
64,0.716,5.0,203.0,0.025,81.0,279.0,0.290
128,0.801,16.0,180.0,0.089,48.0,213.0,0.225
192,0.834,22.0,212.0,0.104,31.0,222.0,0.140
384,0.777,61.0,221.0,0.276,14.0,174.0,0.080
512,0.727,68.0,198.0,0.343,17.0,147.0,0.116
